# Лабораторная работа №3. Анализ сетевого трафика

## Часть 3. Индивидуальное задание

## Выполнили работу:
### Симахин Степан Андреевич ИНБс-5301-01-00
### Игнатович Вадим Олегович ИНБс-5301-01-00
---
<b> Вариант работы: №26. </b>
Задание по варианту:

```
Вариант 26. Bagging деревьев решений. В блокноте 02 обучите
BaggingClassifier(estimator=DecisionTreeClassifier(random_state=2), n_estimators=20, random_state=2,
n_jobs=-1). Сравните accuracy на внутреннем разбиении и на KDDTest+ с одиночным деревом.
Уменьшился ли разрыв между «внутренней» и честной оценками?
```


Ход работы:
* Шаг 1   – подготовка датасетов.
* Шаг 2   – реализация алгоритмов.
  * Шаг 2.1 – реализация алгоритмов одиночного дерева и BaggingTree на внутреннем разбиении, сравнение полученных результатов.
  * Шаг 2.2 – реализация алгоритмов одиночного дерева и BaggingTree на датасетах `KDDTrain+`, сравнение полученных результатов.
  * Шаг 2.3 – сравнение алгоритмов одиночного дерева и BaggingTree на датасете `KDDTest+`, сравнение полученных результатов.
* Шаг 3 – выводы, конечное сравнение.
  * Шаг 3.1. – сравнение результатов при внутреннем разбиении;
  * Шаг 3.2. – сравнение результатов на датасете `KDDTest+`;
  * Шаг 3.3. – общие выводы.


## Шаг 1. Подключение и настройка датасета

Подключение архива с файлами датасета происходит посредством монтирования файла в Google Drive:

In [ ]:
# функция для нормального вывода списков:
def to_str(inp, sep=', '):
  #return str(inp)[1:-1]
  return sep.join(str(it) for it in inp)

# подключение архива datasets.zip, который лежит в Google Диск:
#   прямая ссылка на datasets.zip:
#   https://drive.google.com/file/d/1pt9PgKb9ckkTVfx5YhvzmhfOCREtySsk/view?usp=sharing
from google.colab import drive
drive.mount('/content/drive')
!unzip -q "/content/drive/MyDrive/datasets.zip" -d /content/
print('Файл \'datasets.zip\' подключён')

Mounted at /content/drive
Файл 'datasets.zip' подключён


Подключение файлов в архиве датасета:

In [ ]:
import os

# Где лежат данные: рядом с блокнотом (локально) или в /content (Colab)
# DATA_DIR = 'datasets' if os.path.isdir('datasets') else '/content/datasets'
DATA_DIR = 'datasets'

for name in ['ids.csv', 'KDDTrain+.csv', 'KDDTest+.csv']:
    path = os.path.join(DATA_DIR, name)
    assert os.path.isfile(path), (
        f'Файл {name} не найден по пути {path}. '
        'В Colab выполните ячейку «Шаг 0», локально — положите папку datasets рядом с блокнотом.')

print('Успешно найденные файлы датасета: {}.'.format(to_str(sorted(os.listdir(DATA_DIR)))))

Успешно найденные файлы датасета: KDDTest+.csv, KDDTrain+.csv, ids.csv.


Подключение библиотек:

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import BaggingClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report,
                             ConfusionMatrixDisplay)
from matplotlib.colors import LinearSegmentedColormap

print('Библиотеки подключены')

# what is that below?
RANDOM_STATE = 2  # фиксируем, чтобы результаты воспроизводились

Библиотеки подключены


## Шаг 2. Реализация алгоритмов

## Шаг 2.1. Реализация алгоритмов при внутренней выборке

Подготовка данных для алгоритмов:

In [ ]:
# Чтение файла ids.csv:
df = pd.read_csv(os.path.join(DATA_DIR, 'ids.csv'))

# Получение константных параметров:
const_cols = [c for c in df.columns if df[c].nunique() <= 1]
# Удаление константных параметров:
df = df.drop(columns=const_cols)

# Формирования столбцов параметров:
counts = df['class'].value_counts()

# Формирование traint и test частей:
X = df.drop(columns=['class'])
y = df['class']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.7, random_state=RANDOM_STATE, stratify=y)

# Кодирование и масштабирование:
cat_cols = X.select_dtypes(exclude='number').columns.tolist()
num_cols = X.select_dtypes(include='number').columns.tolist()

# создание DataFrame, который хранит кодированные и масштабированные значения:
preprocess = ColumnTransformer ([
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
    ('num', MinMaxScaler(), num_cols),
])

print('Если ошибок не было, то подготовка данных прошла успешно')

Если ошибок не было, то подготовка данных прошла успешно


Реализация алгоритмов:

In [ ]:
# WARNING: takes less than 1 minute to run

max_depth_list = [8, 15, None]

grid_tree = GridSearchCV(
    make_pipeline(preprocess, DecisionTreeClassifier(random_state=RANDOM_STATE)),
    {'decisiontreeclassifier__max_depth': max_depth_list},
    cv=5, n_jobs=-1, scoring='neg_log_loss')
grid_tree_2 = GridSearchCV(
    make_pipeline(
        preprocess,
        BaggingClassifier(
            estimator=DecisionTreeClassifier(random_state=2),
            n_estimators=20,
            random_state=2,
            n_jobs=-1
        )
    ),
    {'baggingclassifier__estimator__max_depth': max_depth_list},
    cv=5, n_jobs=-1, scoring='neg_log_loss')

t0 = time.time()
grid_tree.fit(X_train, y_train)
t1 = time.time()

t0_2 = time.time()
grid_tree_2.fit(X_train, y_train)
t1_2 = time.time()

print(f'Размер сетки для GridSearchCV = {len(max_depth_list)}')
print(f'Число отбираемых моделей равно 1x{len(max_depth_list)}={1*len(max_depth_list)}')

print(f'\nВремя моделирования всех моделей одинарного дерева: {t1 - t0:.1f} с')
print(f'Параметры лучшей модели:\n\t- \'max_depth\' = {grid_tree.best_params_.get('decisiontreeclassifier__max_depth')}.')
print(f'Значение ошибки на кросс-валидации: {grid_tree.best_score_:.4f}')

print(f'\nВремя моделирования всех моделей BaggingTree: {t1_2 - t0_2:.1f} с')
print(f'Параметры лучшей модели:\n\t- \'max_depth\' = {grid_tree_2.best_params_.get('baggingclassifier__estimator__max_depth')}.')
print(f'Значение ошибки на кросс-валидации: {grid_tree_2.best_score_:.4f}')

Размер сетки для GridSearchCV = 3
Число отбираемых моделей равно 1x3=3

Время моделирования всех моделей одинарного дерева: 8.3 с
Параметры лучшей модели:
	- 'max_depth' = 8.
Значение ошибки на кросс-валидации: -0.1048

Время моделирования всех моделей BaggingTree: 17.9 с
Параметры лучшей модели:
	- 'max_depth' = 8.
Значение ошибки на кросс-валидации: -0.0194


Анализ полученных данных при переходе от одиночного дерева к BaggingTree:
* время моделирования стало значительно больше;
* значение ошибки на кросс-валидации значительно уменьшилось, а значит, модель показывает себя лучше.

## Шаг 2.2. Реализация алгоритмов на `KDDTrain+`

Подготовка данных для алгоритмов:

In [ ]:
# Формирование столбцов:
COLS = ['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
        'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
        'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
        'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
        'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
        'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
        'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
        'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
        'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
        'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty']

# Получение файла KDDTrain+.csv:
data = pd.read_csv(os.path.join(DATA_DIR, 'KDDTrain+.csv'), header=None, names=COLS)

# Категоризирование классов:
ATTACK_CATEGORY = {
    # normal:
    'normal': 'normal',
    # DoS:
    'neptune': 'dos', 'smurf': 'dos', 'back': 'dos', 'teardrop': 'dos',
    'pod': 'dos', 'land': 'dos', 'apache2': 'dos', 'udpstorm': 'dos',
    'processtable': 'dos', 'mailbomb': 'dos', 'worm': 'dos',
    # probing:
    'satan': 'probe', 'ipsweep': 'probe', 'portsweep': 'probe',
    'nmap': 'probe', 'mscan': 'probe', 'saint': 'probe',
    # R2L (remote to local):
    'warezclient': 'r2l', 'guess_passwd': 'r2l', 'warezmaster': 'r2l',
    'imap': 'r2l', 'ftp_write': 'r2l', 'multihop': 'r2l', 'phf': 'r2l',
    'spy': 'r2l', 'httptunnel': 'r2l', 'snmpguess': 'r2l',
    'snmpgetattack': 'r2l', 'named': 'r2l', 'sendmail': 'r2l',
    'xlock': 'r2l', 'xsnoop': 'r2l',
    # U2R (user to root):
    'buffer_overflow': 'u2r', 'rootkit': 'u2r', 'loadmodule': 'u2r',
    'perl': 'u2r', 'ps': 'u2r', 'sqlattack': 'u2r', 'xterm': 'u2r',
}
data['category'] = data['label'].map(ATTACK_CATEGORY)
assert data['category'].notna().all(), 'Найдены метки вне словаря'
cat_counts = data['category'].value_counts()

# Предобработка:
X = data.drop(columns=['label', 'difficulty', 'category'])
y = data['category']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
cat_cols = ['protocol_type', 'service', 'flag']
num_cols = [c for c in X.columns if c not in cat_cols]
preprocess = ColumnTransformer([
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
    ('num', MinMaxScaler(), num_cols),
])

print('Если ошибок не было, то подготовка данных прошла успешно')

Если ошибок не было, то подготовка данных прошла успешно


Реализация алгоритмов при работе с `KDD`:

In [ ]:
# WARNING: takes 3 minutes to run

#max_depth_list = [2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, None]
max_depth_list = [8, 15, None]

grid_tree = GridSearchCV(
    make_pipeline(preprocess, DecisionTreeClassifier(random_state=RANDOM_STATE)),
    {'decisiontreeclassifier__max_depth': max_depth_list},
    cv=5, n_jobs=-1, scoring='neg_log_loss')
grid_tree_2 = GridSearchCV(
    make_pipeline(
        preprocess,
        BaggingClassifier(
            estimator=DecisionTreeClassifier(random_state=2),
            n_estimators=20,
            random_state=2,
            n_jobs=-1)),
    {'baggingclassifier__estimator__max_depth': max_depth_list},
    cv=5, n_jobs=-1, scoring='neg_log_loss')

t0 = time.time()
grid_tree.fit(X_train, y_train)
t1 = time.time()

t0_2 = time.time()
grid_tree_2.fit(X_train, y_train)
t1_2 = time.time()

print(f'Размер сетки для GridSearchCV = {len(max_depth_list)}')
print(f'Число отбираемых моделей равно 1x{len(max_depth_list)}={1*len(max_depth_list)}')

print('\n1. Одномерное дерево:')
print(f'\nВремя моделирования всех моделей: {t1 - t0:.1f} с')
print(f'Параметры лучшей модели:\n\t- \'max_depth\' = {grid_tree.best_params_.get("decisiontreeclassifier__max_depth")}.')
print(f'Значение ошибки на кросс-валидации: {grid_tree.best_score_:.4f}')

print('\n2. BaggingTree:')
print(f'\nВремя моделирования всех моделей: {t1_2 - t0_2:.1f} с')
print(f'Параметры лучшей модели:\n\t- \'max_depth\' = {grid_tree_2.best_params_.get("baggingclassifier__estimator__max_depth")}.')
print(f'Значение ошибки на кросс-валидации: {grid_tree_2.best_score_:.4f}')

Размер сетки для GridSearchCV = 3
Число отбираемых моделей равно 1x3=3

1. Одномерное дерево:

Время моделирования всех моделей: 15.8 с
Параметры лучшей модели:
	- 'max_depth' = 8.
Значение ошибки на кросс-валидации: -0.0529

2. BaggingTree:

Время моделирования всех моделей: 204.5 с
Параметры лучшей модели:
	- 'max_depth' = 15.
Значение ошибки на кросс-валидации: -0.0137


В итоге получены следующие результаты при переходе к BaggingTree:
* выбор другого значения параметра `max_depth`;
* уменьшение ошибки на кросс-валидации;
* значительное увеличение времени обучения.

Сравнение реализаций алгоритмов:

In [ ]:
model_tree = grid_tree.best_estimator_
y_pred = model_tree.predict(X_test)

model_tree_2 = grid_tree_2.best_estimator_
y_pred_2 = model_tree_2.predict(X_test)

print('1. Алгоритм одинарного дерева:')
print(f'\nПолученное значение accuracy на датасете тестирования: {accuracy_score(y_test, y_pred_2):.3f}')
print('Полученные характеристики:\n')
print(classification_report(y_test, y_pred, digits=3))

print('\n2. Алгоритм BaggingTree:')
print(f'\nПолученное значение accuracy на датасете тестирования: {accuracy_score(y_test, y_pred_2):.3f}')
print('Полученные характеристики:\n')
print(classification_report(y_test, y_pred_2, digits=3))

1. Алгоритм одинарного дерева:

Полученное значение accuracy на датасете тестирования: 0.997
Полученные характеристики:

              precision    recall  f1-score   support

         dos      0.996     0.999     0.998      9186
      normal      0.994     0.995     0.994     13469
       probe      0.979     0.976     0.977      2331
         r2l      0.924     0.789     0.851       199
         u2r      0.333     0.100     0.154        10

    accuracy                          0.993     25195
   macro avg      0.845     0.772     0.795     25195
weighted avg      0.992     0.993     0.992     25195


2. Алгоритм BaggingTree:

Полученное значение accuracy на датасете тестирования: 0.997
Полученные характеристики:

              precision    recall  f1-score   support

         dos      0.999     0.999     0.999      9186
      normal      0.997     0.999     0.998     13469
       probe      0.997     0.991     0.994      2331
         r2l      0.978     0.915     0.945       199
   

На данном этапе можно заключить:
* алгоритмы дают одинаково близкий к 100% точности результаты на всех параметрах и классах, кроме u2r и r2l (редко встречающиеся классы);
* для реального анализа моделей необходимо провести тестирование на `KDDTest+`.

## 2.3 Сравнение алгоритмов на `KDDTest+`

Подготовка:

In [ ]:
test_data = pd.read_csv(os.path.join(DATA_DIR, 'KDDTest+.csv'), header=None, names=COLS)
train_data_demo = pd.read_csv(os.path.join(DATA_DIR, 'KDDTrain+.csv'), header=None, names=COLS)
test_data['category'] = test_data['label'].map(ATTACK_CATEGORY)
train_data_demo['category'] = train_data_demo['label'].map(ATTACK_CATEGORY)
assert test_data['category'].notna().all()

new_attacks = sorted(set(test_data['label']) - set(data['label']))

print('Если ошибок не было, то подготовка данных прошла успешно')

Если ошибок не было, то подготовка данных прошла успешно


Реализация алгоритмов:

In [ ]:
# Переобучаем обе модели на ВСЁМ тренировочном наборе и проверяем на KDDTest+
# WARNING: takes 0.5 minutes to run
y = data['category']
X_full, y_full = X, y
X_test_real, y_test_real = (test_data.drop(columns=['label', 'difficulty', 'category']),
                            test_data['category'])

models = [
    ('Дерево решений',
     lambda: DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ('BaggingClassifier (деревья)',
     lambda: BaggingClassifier(
         estimator=DecisionTreeClassifier(random_state=2),
         n_estimators=20,
         random_state=2,
         n_jobs=-1
     )),
]
models_real = {}
for name, make_model in models:
    model = make_pipeline(preprocess, make_model())
    t0 = time.time()
    model.fit(X_full, y_full)
    acc = model.score(X_test_real, y_test_real)
    models_real[name] = model
    print(f'Время обучение модели \'{name}\': {time.time() - t0:.1f} с\nЗначение accuracy на KDDTest+: {acc:.3f}\n')

Время обучение модели 'Дерево решений': 5.8 с
Значение accuracy на KDDTest+: 0.769

Время обучение модели 'BaggingClassifier (деревья)': 31.4 с
Значение accuracy на KDDTest+: 0.758



По полученным данным можно заключить:
* время обучения при BaggingClassifier значительно выше;
* разница в значениях `accuracy` сравнительно небольшая.


Оценка характеристик полученных моделей:

## Шаг 3. Вывод, сравнение алгоритмов

## Шаг 3.1. Сравнение результатов при внутреннем разбиении:

In [ ]:
model_tree = grid_tree.best_estimator_
y_pred = model_tree.predict(X_test)

model_tree_2 = grid_tree_2.best_estimator_
y_pred_2 = model_tree_2.predict(X_test)

print('1. Алгоритм одинарного дерева:')
print(f'\nПолученное значение accuracy на датасете тестирования: {accuracy_score(y_test, y_pred_2):.3f}')
print('Полученные характеристики:\n')
print(classification_report(y_test, y_pred, digits=3))

print('\n2. Алгоритм BaggingTree:')
print(f'\nПолученное значение accuracy на датасете тестирования: {accuracy_score(y_test, y_pred_2):.3f}')
print('Полученные характеристики:\n')
print(classification_report(y_test, y_pred_2, digits=3))

1. Алгоритм одинарного дерева:

Полученное значение accuracy на датасете тестирования: 0.997
Полученные характеристики:

              precision    recall  f1-score   support

         dos      0.996     0.999     0.998      9186
      normal      0.994     0.995     0.994     13469
       probe      0.979     0.976     0.977      2331
         r2l      0.924     0.789     0.851       199
         u2r      0.333     0.100     0.154        10

    accuracy                          0.993     25195
   macro avg      0.845     0.772     0.795     25195
weighted avg      0.992     0.993     0.992     25195


2. Алгоритм BaggingTree:

Полученное значение accuracy на датасете тестирования: 0.997
Полученные характеристики:

              precision    recall  f1-score   support

         dos      0.999     0.999     0.999      9186
      normal      0.997     0.999     0.998     13469
       probe      0.997     0.991     0.994      2331
         r2l      0.978     0.915     0.945       199
   

## Выводы по реализации алгоритмов при внутреннем разбиении

При переходе от алгоритма одинарного дерева к алгоритму BaggingTree полученные значения оказались крайне близки.

Причины данного результата:
* при внутреннем разбиении датасеты для обучения и тестирования имеют равномерное распределение классов;
* принципы работы алгоритмов во многом схожи.

## Шаг 3.2. Сравнение результатов при работе с `KDDTest+`:

In [ ]:
y_pred_real = models_real['Дерево решений'].predict(X_test_real)
print('Подробный отчёт по одиночному дереву:')
print(classification_report(y_test_real, y_pred_real, digits=3, zero_division=0))
y_pred_real = models_real['BaggingClassifier (деревья)'].predict(X_test_real)
print('\nПодробный отчёт по дереву BaggingClassifier:')
print(classification_report(y_test_real, y_pred_real, digits=3, zero_division=0))

Подробный отчёт по одиночному дереву:
              precision    recall  f1-score   support

         dos      0.954     0.828     0.886      7460
      normal      0.700     0.966     0.811      9710
       probe      0.642     0.634     0.638      2421
         r2l      0.941     0.083     0.152      2885
         u2r      0.571     0.179     0.273        67

    accuracy                          0.769     22543
   macro avg      0.762     0.538     0.552     22543
weighted avg      0.808     0.769     0.732     22543


Подробный отчёт по дереву BaggingClassifier:
              precision    recall  f1-score   support

         dos      0.958     0.808     0.877      7460
      normal      0.667     0.968     0.790      9710
       probe      0.766     0.653     0.705      2421
         r2l      0.914     0.022     0.043      2885
         u2r      0.750     0.134     0.228        67

    accuracy                          0.758     22543
   macro avg      0.811     0.517     0.528    

По сравнению можно заключить результаты перехода к BaggingClassifier:
* небольшое увеличение precision по dos и значительное увеличение по u2r и probe;
* небольшое падение precision по normal, r2l;
* значительное увеличение f1-score по u2r.

## Шаг 3.3. Общие выводы

При переходе от алгоритма одиночного дерева к алгоритму BaggingTree получены результаты:
1. Время обучения заметно больше.
2. Значение ошибки при моделировании становится меньше.
3. Оптимальное значение параметра `max_depth` при внутреннем разбиении не меняется, но при работе с `KDDTrain+` оказывается больше.
4. Метрики precision, recall, f1 при работе с редкими категориями (r2l, u2r) принимают более высокие значения.

Сравнивая полученные результаты, можно заключить:
* алгоритм одиночного дерева (DecisionTree) является заметно более быстрым в плане обучения алгоритмом, при этом значения его оценки устойчивы и высоки*; алгоритм может применяться при необходимости получить хорошие и устойчивые значения при минимальных временных затратах;
* алгоритм BaggingTree является заметно более медленным, при этом значения более устойчивы и высоки, особенно при работе с редкими классами/категориями, при этом алгоритм теоретически более устойчив к переобучению и при обучении позволяет получить меньшее значение ошибки; алгоритм может применяться при необходимости получить более хорошие и устойчивые значения при более высоких временных затратах.